# Amazn AI - Day 2
## Embedding Generation & FAISS Vector Store

## Step 1: Import Libraries

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DELL\AppData\Local\Temp\ipykernel_11188\1386063326.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 2: Load Cleaned Dataset

In [2]:
df=pd.read_csv('../data/processed/amazon_cleaned.csv')
df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,"['computers&accessories', 'accessories&periphe...",399.0,1099.0,64.0,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,"['computers&accessories', 'accessories&periphe...",199.0,349.0,43.0,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,"['computers&accessories', 'accessories&periphe...",199.0,1899.0,90.0,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,"['computers&accessories', 'accessories&periphe...",329.0,699.0,53.0,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,"['computers&accessories', 'accessories&periphe...",154.0,399.0,61.0,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


## Step 3: Load Embedding Model

In [3]:
model=SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded successfully!')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 945.79it/s]


Embedding model loaded successfully!


## Step 4: Create Product Documents

In [4]:
documents=[]
for _,row in df.iterrows():
    doc=f"""Product Name: {row['product_name']}

Category: {row['category']}

Discounted Price: ₹{row['discounted_price']}

Actual Price: ₹{row['actual_price']}

Discount: {row['discount_percentage']}%

Rating: {row['rating']}

Description:
{row['about_product']}
"""
    documents.append(doc)
print(len(documents))

1465


## Step 5: Generate Embeddings

In [5]:
embeddings=model.encode(documents,show_progress_bar=True)
print(embeddings.shape)

Batches: 100%|██████████| 46/46 [03:33<00:00,  4.63s/it]

(1465, 384)


## Step 6: Create LangChain Documents

In [6]:
langchain_docs=[]
for i,row in df.iterrows():
    langchain_docs.append(Document(page_content=documents[i],metadata={
        'chunk_id':i,
        'product_name':row['product_name'],
        'category':row['category'],
        'discounted_price':row['discounted_price'],
        'actual_price':row['actual_price'],
        'discount_percentage':row['discount_percentage'],
        'rating':row['rating'],
        'rating_count':row['rating_count']
    }))
print(len(langchain_docs))

1465


## Step 7: Build and Save FAISS

In [7]:
embedding_model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store=FAISS.from_documents(langchain_docs,embedding_model)
print(vector_store.index.ntotal)
vector_store.save_local('../vector_store/faiss_index')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 552.98it/s]


1465


## Step 8: Test Retrieval

In [8]:
query='Laptop with 16GB RAM and 512GB SSD'
results=vector_store.similarity_search(query,k=5)
for i,r in enumerate(results):
    print('='*50)
    print(i+1,r.metadata['product_name'])
    print(r.metadata['category'])
    print(r.metadata['discounted_price'])
    print(r.metadata['rating'])

1 Lenovo IdeaPad 3 11th Gen Intel Core i3 15.6" FHD Thin & Light Laptop(8GB/512GB SSD/Windows 11/Office 2021/2Yr Warranty/3months Xbox Game Pass/Platinum Grey/1.7Kg), 81X800LGIN
['computers&accessories', 'laptops', 'traditionallaptops']
37247.0
4.0
2 Western Digital WD Green SATA 240GB Internal SSD Solid State Drive - SATA 6Gb/s 2.5 inches - WDS240G3G0A
['computers&accessories', 'components', 'internalsolidstatedrives']
1709.0
4.4
3 Crucial BX500 240GB 3D NAND SATA 6.35 cm (2.5-inch) SSD (CT240BX500SSD1)
['computers&accessories', 'components', 'internalsolidstatedrives']
1815.0
4.5
4 Crucial RAM 8GB DDR4 3200MHz CL22 (or 2933MHz or 2666MHz) Laptop Memory CT8G4SFRA32A
['computers&accessories', 'components', 'memory']
1792.0
4.5
5 Lapster Caddy for ssd and HDD, Optical Bay 2nd Hard Drive Caddy, Caddy 9.5mm for Laptop
['computers&accessories', 'components', 'internalharddrives']
199.0
4.2


# Day 2 Complete
Generated embeddings, created FAISS vector store, and tested semantic search.